# 🍃 Leaf Disease Detection — Train YOLOv12 (Good vs Bad Leaf)

This notebook trains a **YOLOv12** object-detection model on your Roboflow dataset and exports it to **ONNX** so it can run 100% in the browser on the GitHub Pages website.

**Steps:** GPU check → install → download dataset → train → evaluate → preview predictions → export ONNX → download.

> **Before you run:** Set the runtime to GPU — `Runtime ▸ Change runtime type ▸ T4 GPU`.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies
`ultralytics` (YOLOv12), `roboflow` (dataset), and `onnx`/`onnxslim` (export).

In [ ]:
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime

import ultralytics
ultralytics.checks()

## 3. Download the dataset from Roboflow
This is your snippet. It downloads the dataset in YOLOv12 format and prints its location.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="LBmxCax9KBgA0ymXpmrL")
project = rf.workspace("sanojs-workspace-uhhjv").project("my-first-project-un4mw")
version = project.version(1)
dataset = version.download("yolov12")

print("Dataset location:", dataset.location)

## 4. Inspect the dataset
Print the class names and counts. **⚠️ Note the class names printed below — you'll paste them into `app.js` on the website.**

In [ ]:
import yaml, os

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

names = cfg["names"]
print("Number of classes:", len(names))
print("Class names (in order):", names)
print("\n👉 Copy this line into app.js  ->  const CLASSES =", list(names) if isinstance(names, dict) else names)

## 5. Train YOLOv12-nano
**Nano** (`yolo12n`) is chosen on purpose: the model runs in the visitor's browser, so it must be small (~6 MB) and fast.

Tune `epochs` if you have more/less time. 100 is a good start; watch for early-stopping.

In [ ]:
from ultralytics import YOLO

# yolo12n.pt = YOLOv12 nano pretrained weights (Ultralytics drops the 'v' for v11/v12)
model = YOLO("yolo12n.pt")

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # early stop if no improvement for 20 epochs
    name="leaf_good_bad",
    plots=True,
)

print("Best weights:", model.trainer.best)

## 6. Evaluate on the validation set

In [ ]:
best = YOLO(model.trainer.best)   # load best.pt
metrics = best.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)

## 7. View training curves & confusion matrix

In [ ]:
from IPython.display import Image, display
import glob, os

run_dir = os.path.dirname(model.trainer.best).replace("/weights", "")
for img in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = os.path.join(run_dir, img)
    if os.path.exists(p):
        print(img)
        display(Image(filename=p, width=700))

## 8. Quick sanity-check prediction
Run the model on a few validation images to see the boxes.

In [ ]:
import glob, os
from IPython.display import Image, display

val_images = glob.glob(os.path.join(dataset.location, "valid", "images", "*"))[:3]
if not val_images:
    val_images = glob.glob(os.path.join(dataset.location, "test", "images", "*"))[:3]

pred = best.predict(val_images, conf=0.25, save=True)
for r in pred:
    display(Image(filename=r.save_dir + "/" + os.path.basename(r.path), width=500))

## 9. Export to ONNX (for the browser)
`opset=12` keeps it compatible with **onnxruntime-web**. `simplify=True` shrinks the graph.

The exported file lives next to `best.pt` as `best.onnx`.

In [ ]:
onnx_path = best.export(format="onnx", opset=12, imgsz=640, simplify=True, dynamic=False)
print("Exported ONNX:", onnx_path)

## 10. Download the trained model
Downloads `best.onnx` (for the website) and `best.pt` (backup). 

➡️ Put **`best.onnx`** into the website's `model/` folder and rename it if needed (the website expects `model/best.onnx`).

In [ ]:
from google.colab import files
import shutil

# Copy to simple names in the working dir, then download
shutil.copy(str(onnx_path), "best.onnx")
shutil.copy(str(model.trainer.best), "best.pt")

files.download("best.onnx")
files.download("best.pt")

---
### ✅ Next steps
1. Drop `best.onnx` into your website repo at `model/best.onnx`.
2. Open `app.js` and set `CLASSES` to the class names printed in **Step 4** (order matters!).
3. Push to GitHub and enable **Settings ▸ Pages**. Done — public URL for everyone.